In [1]:
import os
import json
from pathlib import Path
import numpy as np
import requests
import pandas as pd

from dw_helpers import ensure_season_charts, publish, check, BASE_URL
from dw_helpers import headers as dw_headers
from fantrax_helpers import resolve_latest_gw, fetch_standings, fetch_matchups

In [2]:
# Pick a season with SEASON (e.g. `make update SEASON=16`); default is the newest in seasons.json.
# To start a new season, add its Fantrax league ID to seasons.json -- that's it.
SEASONS = json.loads(Path('seasons.json').read_text())
season = os.environ.get('SEASON', '').strip() or max(SEASONS, key=int)
league_id = SEASONS[season]['league_id']
template_season = str(int(season) - 1)  # a new season's Datawrapper charts are copied from last season's

# Auto-detects the last gameweek Fantrax has scored. To force one, set LATEST_GW,
# e.g. `make update LATEST_GW=10`
latest_gw = resolve_latest_gw(league_id)
print(f'Season {season}, league {league_id}')

latest_gw = 10 (auto-detected)
Season 16, league ag5qep8dms84zona


In [3]:
# All Fantrax requests live in fantrax_helpers.py
df, team_info = fetch_matchups(league_id, latest_gw)

In [4]:
# list of all teams
teams = sorted(df["home_team"].unique())

# Step 1 — Determine winner/loser
def get_result(row):
    if row["away_score"] > row["home_score"]:
        return row["away_team"], row["home_team"]
    elif row["home_score"] > row["away_score"]:
        return row["home_team"], row["away_team"]
    else:
        return None, None  # tie

df["winner"], df["loser"] = zip(*df.apply(get_result, axis=1))

# Step 2 — Build win records
wins = df.dropna(subset=["winner"]).loc[:, ["winner", "loser"]]
wins["count"] = 1

# Step 3 — Aggregate counts
win_counts = (
    wins.groupby(["winner", "loser"])["count"]
        .sum()
        .reset_index()
)

# Step 4 — Ensure all teams appear as losers for every winner
all_pairs = pd.MultiIndex.from_product([teams, teams], names=["winner", "loser"]).to_frame(index=False)
win_counts = all_pairs.merge(win_counts, on=["winner", "loser"], how="left").fillna(0).rename(
    columns={'count':'wins'}
)
win_counts["wins"] = win_counts["wins"].astype(int)

In [5]:
# Create mapping from team to index
team_index = {team: i for i, team in enumerate(teams)}

# Add x/y coordinates
win_counts["x"] = win_counts["loser"].map(team_index)
win_counts["y"] = win_counts["winner"].map(team_index)

In [6]:
# blank out the diagonal (a team vs. itself); the column has to hold text for that
win_counts["wins"] = win_counts["wins"].astype(object)
win_counts.loc[win_counts["winner"] == win_counts["loser"], "wins"] = ""

In [7]:
win_counts.to_csv(f'data/output/h2h/season-{season}.csv', index=False)

In [8]:
chart_data = win_counts.to_csv(index=False)

In [9]:
name_to_short = {v["name"]: v.get("shortName") for v in team_info.values()}
# fall back to initials for any team without a Fantrax short name
for t in teams:
    if not name_to_short.get(t):
        name_to_short[t] = ''.join(w[0] for w in t.split() if w[:1].isalnum()).upper()[:4]

In [10]:
name_to_short.setdefault("Mattchester United", "MU")

'Champion'

In [25]:
name_to_short["Mattchester United"] = 'CHAMP'

In [26]:
name_to_short

{'Cheasle FC': 'CFC',
 'NikkiPSG FC': 'NPSG',
 'Mattchester United': 'CHAMP',
 'Seanhampton FC': 'SHFC',
 'ZacBurn Rovers FC': 'ZAC',
 "Schadenasty's (Rafa)": 'RAFA',
 'Thottenham Hotsluts': 'THOTS',
 'Newcastle UN-HINGED': 'NUF🖕🏾',
 'Benford FC': 'BFC',
 'West Thom United': 'WTU'}

In [27]:
season_charts = ensure_season_charts(season, template_season)
chart_id = season_charts['h2h']['chart_id']

chart_url = f'{BASE_URL}/{chart_id}'
chart_data_url = f'{chart_url}/data'
json_headers = dw_headers()
csv_headers = dw_headers('text/csv')

Send latest data to Datawrapper

In [28]:
update_data_response = requests.put(chart_data_url, data=chart_data, headers=csv_headers)

Generate x-axis annotations

In [29]:
x_axis_annotions = []
for name, index in team_index.items():
    short_name = name_to_short[name]    
    anno_obj = {
        'dx': 0,
        'dy': 0,
        'id': f'x-{short_name}-{index}',
        'size': 10,
        'text': short_name,
        'align': 'bc',
        'width': 12.5,
        'position': {'x': index, 'y': '-0.5'},
    }
    x_axis_annotions.append(anno_obj)

Generate y-axis annotations

In [30]:
y_axis_annotions = []
for name, index in team_index.items():
    short_name = name_to_short[name]    
    anno_obj = {
        'dx': 0,
        'dy': 0,
        'id': f'y-{short_name}-{index}',
        'size': 10,
        'text': short_name,
        'align': 'mr',
        'width': 12.5,
        'position': {'x': '-0.55', 'y': index},
    }
    y_axis_annotions.append(anno_obj)

Set color scale responsive to max no. of wins

In [31]:
max_wins = max_wins = win_counts['wins'].where(win_counts['wins'] != '', 0).astype(int).max()
max_wins

2

In [32]:
WIN_COLORSETS = {
    0: ['#ffffcc'],
    1: ['#ffffcc', '#c2e699'],
    2: ['#ffffcc', '#c2e699', '#78c679'],
    3: ['#ffffcc', '#c2e699', '#78c679', '#238443'],
    4: ['#ffffcc', '#c2e699', '#78c679', '#238443', '#006837'],
    5: ['#ffffcc', '#d9f0a3', '#addd8e', '#78c679', '#31a354', '#006837'],
}

DEFAULT_COLORSET = ['#f7fcb9', '#addd8e', '#31a354']

def make_color_scheme(max_wins):
    colors = WIN_COLORSETS.get(max_wins, DEFAULT_COLORSET)

    color_map = {str(i): colors[i] for i in range(len(colors))}
    color_map[""] = "#ffffff"

    return {
        "map": color_map,
        "categoryLabels": {
            str(i): f"{i} wins" if i == max_wins and max_wins > 1 else str(i)
            for i in range(len(colors))
        }
    }

color_categories = make_color_scheme(max_wins)

Send to Datawrapper

In [33]:
payload = {
    "metadata": {
        "visualize": {
            "color-category": color_categories,
            "text-annotations": x_axis_annotions + y_axis_annotions
        }
    }
}

In [34]:
response = requests.patch(chart_url, json=payload, headers=json_headers)
response

<Response [200]>

In [35]:
latest_chart_version = publish(chart_id, season, 'h2h')

Published h2h FoX4v v10


In [36]:
form_melted = df[df.gw >= (latest_gw-4)].melt(id_vars=['gw', 'winner', 'loser'], 
                 value_vars=['away_team', 'home_team'], 
                 value_name='team')

# 2. ✓ win / ✗ loss / – draw (en dash) per team per gameweek; Datawrapper colors the cells by these values
def get_form(row):
    if row['winner'] == row['team']:
        return '✓'
    elif row['loser'] == row['team']:
        return '✗'
    else:
        return '–'

form_melted['form'] = form_melted.apply(get_form, axis=1)

In [37]:
form_table = form_melted.pivot(index='team', columns='gw', values='form').sort_index(axis=1)
form_gws = list(form_table.columns)

# Best form first: most wins, then fewest losses, then alphabetical
form_table = form_table.assign(
    _wins=(form_table == '✓').sum(axis=1),
    _losses=(form_table == '✗').sum(axis=1),
    _team=form_table.index.str.lower(),
).sort_values(['_wins', '_losses', '_team'], ascending=[False, True, True]).drop(columns=['_wins', '_losses', '_team'])

# Column names stay the same every week (oldest -> newest) so the cell colors set up
# in Datawrapper, which are tied to column names, keep working. The actual gameweeks
# go in the chart's description instead.
FORM_LABELS = ['-5', '-4', '-3', '-2', '-1', 'Latest']
form_table.columns = FORM_LABELS[-len(form_gws):]
# Fantrax short names (e.g. MU) instead of full team names
form_table.index = form_table.index.map(lambda t: name_to_short.get(t) or t)
form_table.index.name = 'Team'
form_table

,-4,-3,-2,-1,Latest
Team,,,,,
NPSG,✓,✓,✓,✓,✓
BFC,✓,✓,✓,✗,✓
NUF🖕🏾,✓,✓,✗,✓,✓
CHAMP,✗,✓,✓,✓,✗
WTU,✗,✓,✗,✓,✓
RAFA,✓,✗,✓,✗,✗
ZAC,✓,✗,✗,✗,✓
CFC,✗,✗,✗,✓,✗
SHFC,✗,✗,✓,✗,✗


In [ ]:
# Form table: one Datawrapper chart (JePAw) that every run overwrites, not one per season.
# Only the newest season updates it, so `make update SEASON=15` can't clobber current form.
FORM_JSON = Path('../_data/form.json')
form_chart = json.loads(FORM_JSON.read_text())
form_chart_id = form_chart['chart_id']

if season == max(SEASONS, key=int):
    r = requests.put(
        f'{BASE_URL}/{form_chart_id}/data',
        data=form_table.to_csv().encode('utf-8'),  # utf-8 so ✓ ✗ – survive
        headers=dw_headers('text/csv; charset=utf-8'),
    )
    check(r)
    intro = f'Gameweeks {form_gws[0]}–{form_gws[-1]}'

    # Cell colors by result, borrowed from the rest of the site:
    #   win  = a dark green from the head-to-head chart's win scale
    #   loss = the site's red-card coral (--red in styles/app.scss)
    #   draw = the site's light gray ($gray)
    # Colorblind-checked; every symbol meets 4.5:1 contrast on its cell.
    FORM_BACKGROUND = {'✓': '#23844399', '✗': '#FA7F7399', '–': '#F2F2F299'}
    FORM_TEXT = {'✓': '#FFFFFF', '✗': '#4A120C', '–': '#666666'}
    columns = {
        label: {
            'customColor': True,
            'customColorBy': label,  # each column is colored by its own values
            'customColorBackground': FORM_BACKGROUND,
            'customColorText': FORM_TEXT,
        }
        for label in FORM_LABELS
    }

    r = requests.patch(
        f'{BASE_URL}/{form_chart_id}',
        json={'metadata': {
            'describe': {'intro': intro},
            'visualize': {'columns': columns},
        }},
        headers=dw_headers(),
    )
    check(r)
    form_chart['version'] = publish(form_chart_id)
    FORM_JSON.write_text(json.dumps(form_chart, indent=2) + '\n')
else:
    print(f'Season {season} is not the newest season; leaving the form table alone.')